# Experiment 23: Family + Ticket Structure

This experiment tests whether Titanic performance improves when we model family and ticket relationships while keeping survival-derived features leakage-safe.

We compare ordinary StratifiedKFold with Ticket GroupKFold, and compare a passenger/control model against models using out-of-fold group survival encodings.

No external Titanic labels are used and this experiment does not create a submission CSV.

In [14]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

ROOT = Path.cwd().parent
DATA = ROOT / "data"

train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")

TARGET = "Survived"
ID = "PassengerId"
SEED = 42

y = train[TARGET].astype(int).reset_index(drop=True)

print("Train:", train.shape)
print("Test:", test.shape)

Train: (891, 12)
Test: (418, 11)


In [15]:
def engineer(df):
    x = df.copy()

    for c, v in {
        "Name": "Unknown",
        "Ticket": "Unknown",
        "Cabin": "Unknown",
        "Sex": "Unknown",
        "Embarked": "Unknown"
    }.items():
        x[c] = x[c].fillna(v).astype(str)

    x["FamilySize"] = x["SibSp"].fillna(0) + x["Parch"].fillna(0) + 1
    x["IsAlone"] = (x["FamilySize"] == 1).astype(int)
    x["Child"] = (x["Age"] < 16).astype(int)
    x["Mother"] = (
        (x["Sex"] == "female") &
        (x["Parch"] > 0) &
        (x["Age"] > 18)
    ).astype(int)

    x["AgeMissing"] = x["Age"].isna().astype(int)
    x["FareMissing"] = x["Fare"].isna().astype(int)

    x["Title"] = (
        x["Name"]
        .str.extract(r",\s*([^.]*)\.", expand=False)
        .fillna("Unknown")
        .str.strip()
        .replace({
            "Mlle": "Miss",
            "Ms": "Miss",
            "Mme": "Mrs"
        })
    )

    x.loc[
        ~x["Title"].isin(["Mr", "Miss", "Mrs", "Master"]),
        "Title"
    ] = "Rare"

    x["Surname"] = (
        x["Name"]
        .str.split(",")
        .str[0]
        .str.strip()
        .replace("", "Unknown")
    )

    x["TicketPrefix"] = (
        x["Ticket"]
        .str.replace(r"\d", "", regex=True)
        .str.replace(r"[./ ]", "", regex=True)
        .replace("", "NONE")
    )

    x["TicketGroupSize"] = x.groupby("Ticket")["Ticket"].transform("size")
    x["SurnameGroupSize"] = x.groupby("Surname")["Surname"].transform("size")

    x["FarePerPerson"] = (
        x["Fare"] /
        x["TicketGroupSize"].replace(0, np.nan)
    )

    x["SexPclass"] = x["Sex"] + "_" + x["Pclass"].astype(str)
    x["FamilySex"] = x["FamilySize"].astype(str) + "_" + x["Sex"]
    x["PclassTitle"] = x["Pclass"].astype(str) + "_" + x["Title"]

    x["AgeBand"] = pd.cut(
        x["Age"],
        [-np.inf, 12, 18, 30, 50, np.inf],
        labels=False
    ).astype(str)

    x["FareBand"] = pd.cut(
        x["Fare"],
        [-np.inf, 7.5, 15, 30, 60, np.inf],
        labels=False
    ).astype(str)

    x["FamilyTicket"] = x["Surname"] + "_" + x["Ticket"]

    x["NameLength"] = x["Name"].str.len()
    x["NameWords"] = x["Name"].str.split().str.len()
    x["TicketLength"] = x["Ticket"].str.len()

    x["DeckKnown"] = (x["Cabin"] != "Unknown").astype(int)
    x["CabinDeck"] = x["Cabin"].str[0].where(
        x["Cabin"] != "Unknown",
        "Unknown"
    )

    x["LargeFamily"] = (x["FamilySize"] >= 5).astype(int)
    x["SmallFamily"] = x["FamilySize"].between(2, 4).astype(int)

    x["FemaleChild"] = (
        (x["Sex"] == "female") |
        (x["Child"] == 1)
    ).astype(int)

    x["FarePerAge"] = x["Fare"] / x["Age"].clip(lower=1)
    x["ClassFare"] = x["Pclass"] * x["Fare"]
    x["SiblingChildRatio"] = x["SibSp"] / (x["Parch"] + 1)
    x["FamilyFare"] = x["Fare"] * x["FamilySize"]
    x["SexTitle"] = x["Sex"] + "_" + x["Title"]

    return x


allx = engineer(
    pd.concat(
        [train.drop(columns=[TARGET]), test],
        ignore_index=True
    )
)

base = allx.iloc[:len(train)].reset_index(drop=True)
test_base = allx.iloc[len(train):].reset_index(drop=True)

GROUPS = [
    "Ticket",
    "Surname",
    "FamilyTicket"
]

DROP = [
    ID,
    "Name",
    "Ticket",
    "Cabin",
    "Surname",
    "FamilyTicket"
]

print("Engineered train:", base.shape)
print("Engineered test:", test_base.shape)

Engineered train: (891, 42)
Engineered test: (418, 42)


In [16]:
def group_maps(ref, yref, smoothing=5):
    gm = float(pd.Series(yref).mean())
    maps = {}

    for c in GROUPS:
        t = pd.DataFrame({
            "k": ref[c].astype(str).values,
            "y": np.asarray(yref)
        })

        s = t.groupby("k")["y"].agg(["mean", "count"])

        maps[c] = (
            (
                s["mean"] * s["count"] +
                gm * smoothing
            ) /
            (s["count"] + smoothing)
        ).to_dict()

    return maps, gm


def apply_maps(df, maps, gm):
    o = pd.DataFrame(index=df.index)

    for c in GROUPS:
        o[c + "SurvivalTE"] = (
            df[c]
            .astype(str)
            .map(maps[c])
            .fillna(gm)
        )

    return o


def safe_features(tr, ytr, va):
    tr = tr.reset_index(drop=True)
    va = va.reset_index(drop=True)
    ytr = pd.Series(ytr).reset_index(drop=True)

    oof = pd.DataFrame(index=tr.index)

    inner = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED
    )

    for a, b in inner.split(tr, ytr):
        mp, gm = group_maps(
            tr.iloc[a],
            ytr.iloc[a]
        )

        z = apply_maps(
            tr.iloc[b],
            mp,
            gm
        )

        oof.loc[b, z.columns] = z.values

    mp, gm = group_maps(tr, ytr)

    va_group = apply_maps(
        va,
        mp,
        gm
    )

    tr2 = tr.copy()
    va2 = va.copy()

    for c in GROUPS:
        counts = tr[c].astype(str).value_counts()

        tr2[c + "TrainCount"] = (
            tr[c]
            .astype(str)
            .map(counts)
            .fillna(0)
        )

        va2[c + "TrainCount"] = (
            va[c]
            .astype(str)
            .map(counts)
            .fillna(0)
        )

    return (
        pd.concat(
            [tr2, oof],
            axis=1
        ),
        pd.concat(
            [va2, va_group],
            axis=1
        )
    )

In [17]:
def prep(df):
    x = df.drop(
        columns=[c for c in DROP if c in df.columns]
    ).copy()

    for c in x.select_dtypes("object").columns:
        x[c] = x[c].fillna("Unknown").astype(str)

    return x


def models(xtr, ytr, xva):

    # Prepare train and validation data
    a = prep(xtr)
    b = prep(xva)

    # Ensure validation has exactly the same feature columns
    b = b.reindex(columns=a.columns)

    # =========================
    # XGBoost
    # =========================

    cat_cols = a.select_dtypes(
        "object"
    ).columns.tolist()

    num_cols = [
        c for c in a.columns
        if c not in cat_cols
    ]

    pre = ColumnTransformer([
        (
            "num",
            SimpleImputer(
                strategy="median"
            ),
            num_cols
        ),
        (
            "cat",
            Pipeline([
                (
                    "imp",
                    SimpleImputer(
                        strategy="most_frequent"
                    )
                ),
                (
                    "oh",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False
                    )
                )
            ]),
            cat_cols
        )
    ])

    xgb = XGBClassifier(
        n_estimators=900,
        max_depth=3,
        learning_rate=0.025,
        subsample=0.82,
        colsample_bytree=0.85,
        min_child_weight=3,
        gamma=0.05,
        reg_alpha=0.05,
        reg_lambda=2.5,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=SEED,
        n_jobs=-1
    )

    xp = Pipeline([
        ("pre", pre),
        ("model", xgb)
    ])

    xp.fit(
        a,
        ytr
    )

    p1 = xp.predict_proba(
        b
    )[:, 1]

    # =========================
    # CatBoost
    # =========================

    a = a.copy()
    b = b.copy()

    cb_cols = a.select_dtypes(
        "object"
    ).columns.tolist()

    for c in cb_cols:
        a[c] = a[c].fillna("Unknown")
        b[c] = b[c].fillna("Unknown")

    for c in a.columns:

        if c not in cb_cols:

            a[c] = a[c].replace(
                [np.inf, -np.inf],
                np.nan
            )

            b[c] = b[c].replace(
                [np.inf, -np.inf],
                np.nan
            )

            med = a[c].median()

            a[c] = a[c].fillna(med)
            b[c] = b[c].fillna(med)

    cb = CatBoostClassifier(
        iterations=900,
        depth=5,
        learning_rate=0.025,
        l2_leaf_reg=7,
        loss_function="Logloss",
        verbose=False,
        random_seed=SEED,
        thread_count=-1
    )

    cb.fit(
        a,
        ytr,
        cat_features=cb_cols
    )

    p2 = cb.predict_proba(
        b
    )[:, 1]

    return p1, p2


def run_cv(splitter, name, groups=None):

    rows = []

    oof = np.zeros(
        (len(base), 3)
    )

    for fold, (tridx, vidx) in enumerate(
        splitter.split(base, y, groups),
        1
    ):

        print(
            f"{name} fold {fold}/5"
        )

        xtr, xva = safe_features(
            base.iloc[tridx],
            y.iloc[tridx],
            base.iloc[vidx]
        )

        p1, p2 = models(
            xtr,
            y.iloc[tridx],
            xva
        )

        predictions = [
            p1,
            p2,
            0.5 * p1 + 0.5 * p2
        ]

        names = [
            "XGB_group",
            "CAT_group",
            "BLEND_group"
        ]

        for j, p in enumerate(predictions):

            oof[vidx, j] = p

            rows.append({
                "Validation": name,
                "Fold": fold,
                "Model": names[j],
                "Accuracy": accuracy_score(
                    y.iloc[vidx],
                    p >= 0.5
                )
            })

    return pd.DataFrame(rows), oof


# =========================
# Stratified K-Fold
# =========================

sk = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

strat_df, strat_oof = run_cv(
    sk,
    "StratifiedKFold"
)


# =========================
# Ticket Group K-Fold
# =========================

gk = GroupKFold(
    n_splits=5
)

group_df, group_oof = run_cv(
    gk,
    "TicketGroupKFold",
    base["Ticket"].astype(str)
)


# =========================
# Results
# =========================

print()
print("STRATIFIED SUMMARY")

display(
    strat_df
    .groupby("Model")["Accuracy"]
    .agg(
        ["mean", "std", "min", "max"]
    )
    .sort_values(
        "mean",
        ascending=False
    )
)


print()
print("TICKET GROUP SUMMARY")

display(
    group_df
    .groupby("Model")["Accuracy"]
    .agg(
        ["mean", "std", "min", "max"]
    )
    .sort_values(
        "mean",
        ascending=False
    )
)

StratifiedKFold fold 1/5
StratifiedKFold fold 2/5
StratifiedKFold fold 3/5
StratifiedKFold fold 4/5
StratifiedKFold fold 5/5
TicketGroupKFold fold 1/5
TicketGroupKFold fold 2/5
TicketGroupKFold fold 3/5
TicketGroupKFold fold 4/5
TicketGroupKFold fold 5/5

STRATIFIED SUMMARY


,mean,std,min,max
Model,,,,
CAT_group,0.858578,0.016650,0.831461,0.870787
BLEND_group,0.850725,0.014154,0.825843,0.859551
XGB_group,0.845094,0.018671,0.820225,0.865922



TICKET GROUP SUMMARY


,mean,std,min,max
Model,,,,
BLEND_group,0.802467,0.020042,0.780899,0.831461
CAT_group,0.797947,0.029960,0.764045,0.826816
XGB_group,0.794614,0.011605,0.786517,0.814607


In [18]:
# Passenger/control comparison.
# Survival-derived group target encodings are removed.

TARGET_TE = [
    c for c in base.columns
    if c.endswith("SurvivalTE")
]

control = []

for fold, (tridx, vidx) in enumerate(
    sk.split(base, y),
    1
):

    xtr, xva = safe_features(
        base.iloc[tridx],
        y.iloc[tridx],
        base.iloc[vidx]
    )

    xtr = xtr.drop(
        columns=TARGET_TE,
        errors="ignore"
    )

    xva = xva.drop(
        columns=TARGET_TE,
        errors="ignore"
    )

    p1, p2 = models(
        xtr,
        y.iloc[tridx],
        xva
    )

    p = 0.5 * p1 + 0.5 * p2

    score = accuracy_score(
        y.iloc[vidx],
        p >= 0.5
    )

    control.append(score)

    print(
        f"Control fold {fold}: {score:.4f}"
    )

print()
print(
    f"Passenger/control mean: "
    f"{np.mean(control):.4f} +/- "
    f"{np.std(control):.4f}"
)

print()
print("Experiment 23 takeaway:")
print(
    "Compare group-aware and control means. "
    "Treat a large StratifiedKFold advantage that disappears "
    "under TicketGroupKFold as evidence of group leakage/optimism, "
    "not a real improvement."
)

Control fold 1: 0.8547
Control fold 2: 0.8539
Control fold 3: 0.8258
Control fold 4: 0.8596
Control fold 5: 0.8596

Passenger/control mean: 0.8507 +/- 0.0127

Experiment 23 takeaway:
Compare group-aware and control means. Treat a large StratifiedKFold advantage that disappears under TicketGroupKFold as evidence of group leakage/optimism, not a real improvement.


In [19]:
# Final full-data fit for inspection only.
# No CSV is written.
#
# Training target encodings are generated OOF.
# Test encodings use all labeled training rows.

mp, gm = group_maps(
    base,
    y
)

test_group = apply_maps(
    test_base,
    mp,
    gm
)

oof = pd.DataFrame(
    index=base.index
)

inner = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

for a, b in inner.split(base, y):

    m, g = group_maps(
        base.iloc[a],
        y.iloc[a]
    )

    z = apply_maps(
        base.iloc[b],
        m,
        g
    )

    oof.loc[
        b,
        z.columns
    ] = z.values

train_final = base.copy()
test_final = test_base.copy()

for c in GROUPS:

    counts = (
        base[c]
        .astype(str)
        .value_counts()
    )

    train_final[
        c + "TrainCount"
    ] = (
        base[c]
        .astype(str)
        .map(counts)
        .fillna(0)
    )

    test_final[
        c + "TrainCount"
    ] = (
        test_base[c]
        .astype(str)
        .map(counts)
        .fillna(0)
    )

train_final = pd.concat(
    [
        train_final,
        oof
    ],
    axis=1
)

test_final = pd.concat(
    [
        test_final,
        test_group
    ],
    axis=1
)

p1, p2 = models(
    train_final,
    y,
    test_final
)

final_prob = (
    0.5 * p1 +
    0.5 * p2
)

final_pred = (
    final_prob >= 0.5
).astype(int)

print(
    "Test rows:",
    len(final_pred)
)

print(
    "Predicted survived:",
    int(final_pred.sum())
)

print(
    "Predicted not survived:",
    int((1 - final_pred).sum())
)

print(
    "No submission file created in Experiment 23."
)

Test rows: 418
Predicted survived: 149
Predicted not survived: 269
No submission file created in Experiment 23.
